# Northwind Company — End-to-End Sales & Operations Analytics
**Tools used:** Python (pandas) → SQL (sqlite3) → Excel → Power BI

**Business context:** Northwind Traders is a wholesale food & beverage distribution company selling to customers across 21 countries via 9 sales employees, 29 suppliers, and 6 shipping carriers. This project analyzes company-wide sales performance, customer behavior, employee performance, and delivery efficiency to surface actionable business insights.

**Pipeline:**
1. Load & clean raw relational data (Python/pandas)
2. Load into a SQL database and run business queries (sqlite3 + SQL)
3. Export subsets for Excel pivot analysis
4. Feed cleaned data into Power BI for the final dashboard

Data source: Northwind sample database (public domain sample dataset, originally distributed with Microsoft Access, widely used for SQL/BI teaching).

## 1. Setup — Mount Drive & Create Project Folders

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

BASE = '/content/drive/MyDrive/ecommerce-analyst-project'

folders = ['data_raw', 'data_clean', 'sql', 'excel', 'powerbi_export', 'notebooks']
for f in folders:
    os.makedirs(os.path.join(BASE, f), exist_ok=True)

print("Folders ready:")
for f in folders:
    print(" -", os.path.join(BASE, f))

## 2. Get the Data

Upload the 9 CSV files (customers, employees, orders, order-details, products, suppliers, categories, territories, employee-territories) into `data_raw/` in your Drive project folder. Then load them here.

In [ ]:
import pandas as pd

RAW = os.path.join(BASE, 'data_raw')
CLEAN = os.path.join(BASE, 'data_clean')

customers   = pd.read_csv(os.path.join(RAW, 'customers.csv'))
employees   = pd.read_csv(os.path.join(RAW, 'employees.csv'))
orders      = pd.read_csv(os.path.join(RAW, 'orders.csv'))
order_details = pd.read_csv(os.path.join(RAW, 'order-details.csv'))
products    = pd.read_csv(os.path.join(RAW, 'products.csv'))
suppliers   = pd.read_csv(os.path.join(RAW, 'suppliers.csv'))
categories  = pd.read_csv(os.path.join(RAW, 'categories.csv'))

print(customers.shape, employees.shape, orders.shape, order_details.shape, products.shape, suppliers.shape, categories.shape)

## 3. Clean the Data

Northwind is fairly clean already (it's a teaching dataset), but real analyst work always includes a data-quality check. We:
- Check for nulls and duplicates
- Fix date columns to proper datetime type
- Drop columns not needed for analysis (e.g., binary photo/picture blobs)
- Standardize the `region` field (many rows use the literal string `"NULL"` instead of a true null)

In [ ]:
# --- Data quality check ---
for name, df in [('customers', customers), ('orders', orders), ('order_details', order_details),
                  ('products', products), ('employees', employees), ('suppliers', suppliers)]:
    print(f"--- {name} ---")
    print("Nulls:\n", df.isnull().sum()[df.isnull().sum() > 0])
    print("Duplicate rows:", df.duplicated().sum())
    print()

In [ ]:
# --- Fix literal 'NULL' strings -> real NaN ---
for df in [customers, employees, orders, suppliers]:
    df.replace('NULL', pd.NA, inplace=True)

# --- Fix date columns ---
orders['orderDate'] = pd.to_datetime(orders['orderDate'])
orders['requiredDate'] = pd.to_datetime(orders['requiredDate'])
orders['shippedDate'] = pd.to_datetime(orders['shippedDate'], errors='coerce')
employees['birthDate'] = pd.to_datetime(employees['birthDate'])
employees['hireDate'] = pd.to_datetime(employees['hireDate'])

# --- Drop heavy/irrelevant columns for analysis ---
employees_clean = employees.drop(columns=['photo', 'notes', 'photoPath'], errors='ignore')
categories_clean = categories.drop(columns=['picture'], errors='ignore')

# --- Derived column: line revenue per order item ---
order_details['lineRevenue'] = order_details['unitPrice'] * order_details['quantity'] * (1 - order_details['discount'])

# --- Derived column: delivery time in days ---
orders['deliveryDays'] = (orders['shippedDate'] - orders['orderDate']).dt.days
orders['lateDelivery'] = orders['shippedDate'] > orders['requiredDate']

print("Cleaning complete.")
print(order_details[['orderID','productID','unitPrice','quantity','discount','lineRevenue']].head())

In [ ]:
# --- Save cleaned tables ---
customers.to_csv(os.path.join(CLEAN, 'customers_clean.csv'), index=False)
employees_clean.to_csv(os.path.join(CLEAN, 'employees_clean.csv'), index=False)
orders.to_csv(os.path.join(CLEAN, 'orders_clean.csv'), index=False)
order_details.to_csv(os.path.join(CLEAN, 'order_details_clean.csv'), index=False)
products.to_csv(os.path.join(CLEAN, 'products_clean.csv'), index=False)
suppliers.to_csv(os.path.join(CLEAN, 'suppliers_clean.csv'), index=False)
categories_clean.to_csv(os.path.join(CLEAN, 'categories_clean.csv'), index=False)

print("Cleaned CSVs saved to:", CLEAN)

## 4. Load Into SQL (sqlite3)

We now load the cleaned tables into a real SQL database file and query it with actual SQL — not just pandas filtering. This is the section that demonstrates SQL skills on your resume.

**Note:** we run queries with `pd.read_sql_query()` instead of the `%%sql` magic. The `%%sql` magic (via the `ipython-sql` package) frequently breaks in Colab due to version conflicts with a dependency called `prettytable` — you may have hit a `KeyError: 'DEFAULT'` error if you tried it. Using `pd.read_sql_query()` runs the exact same SQL and is actually more reliable, since the result comes back as a normal, readable pandas DataFrame.

In [ ]:
import sqlite3

db_path = os.path.join(BASE, 'northwind.db')
conn = sqlite3.connect(db_path)

customers.to_sql('customers', conn, if_exists='replace', index=False)
employees_clean.to_sql('employees', conn, if_exists='replace', index=False)
orders.to_sql('orders', conn, if_exists='replace', index=False)
order_details.to_sql('order_details', conn, if_exists='replace', index=False)
products.to_sql('products', conn, if_exists='replace', index=False)
suppliers.to_sql('suppliers', conn, if_exists='replace', index=False)
categories_clean.to_sql('categories', conn, if_exists='replace', index=False)

print("Database created at:", db_path)

In [ ]:
def run_query(sql):
    """Run a SQL query against the northwind.db database and return a pandas DataFrame."""
    return pd.read_sql_query(sql, conn)

print("run_query() ready. Example: run_query('SELECT * FROM customers LIMIT 5')")

## 5. Business Queries (SQL)

Each query below answers a real business question. These are the exact same queries saved in `sql/analysis.sql` for your GitHub repo — only how we *run* them (via `run_query()` instead of `%%sql`) is different.

**Q1 — Monthly revenue trend (company-wide)**

In [ ]:
q1 = '''
SELECT
    strftime('%Y-%m', o.orderDate) AS month,
    ROUND(SUM(od.unitPrice * od.quantity * (1 - od.discount)), 2) AS revenue
FROM orders o
JOIN order_details od ON o.orderID = od.orderID
GROUP BY month
ORDER BY month;
'''
run_query(q1)

**Q2 — Top 10 products by revenue**

In [ ]:
q2 = '''
SELECT
    p.productName,
    ROUND(SUM(od.unitPrice * od.quantity * (1 - od.discount)), 2) AS revenue,
    SUM(od.quantity) AS units_sold
FROM order_details od
JOIN products p ON od.productID = p.productID
GROUP BY p.productName
ORDER BY revenue DESC
LIMIT 10;
'''
run_query(q2)

**Q3 — Revenue by country**

In [ ]:
q3 = '''
SELECT
    c.country,
    ROUND(SUM(od.unitPrice * od.quantity * (1 - od.discount)), 2) AS revenue,
    COUNT(DISTINCT o.orderID) AS num_orders
FROM orders o
JOIN customers c ON o.customerID = c.customerID
JOIN order_details od ON o.orderID = od.orderID
GROUP BY c.country
ORDER BY revenue DESC;
'''
run_query(q3)

**Q4 — Employee sales performance ranking (window function)**

In [ ]:
q4 = '''
SELECT
    e.firstName || ' ' || e.lastName AS employee,
    ROUND(SUM(od.unitPrice * od.quantity * (1 - od.discount)), 2) AS revenue,
    RANK() OVER (ORDER BY SUM(od.unitPrice * od.quantity * (1 - od.discount)) DESC) AS sales_rank
FROM orders o
JOIN employees e ON o.employeeID = e.employeeID
JOIN order_details od ON o.orderID = od.orderID
GROUP BY employee
ORDER BY sales_rank;
'''
run_query(q4)

**Q5 — Customer RFM inputs (Recency, Frequency, Monetary)**

In [ ]:
q5 = '''
SELECT
    c.customerID,
    c.companyName,
    JULIANDAY('1998-05-06') - JULIANDAY(MAX(o.orderDate)) AS recency_days,
    COUNT(DISTINCT o.orderID) AS frequency,
    ROUND(SUM(od.unitPrice * od.quantity * (1 - od.discount)), 2) AS monetary
FROM customers c
JOIN orders o ON c.customerID = o.customerID
JOIN order_details od ON o.orderID = od.orderID
GROUP BY c.customerID, c.companyName
ORDER BY monetary DESC
LIMIT 20;
'''
run_query(q5)

**Q6 — Average delivery time & late delivery rate by shipper**

In [ ]:
q6 = '''
SELECT
    o.shipVia AS shipper_id,
    ROUND(AVG(JULIANDAY(o.shippedDate) - JULIANDAY(o.orderDate)), 1) AS avg_delivery_days,
    ROUND(100.0 * SUM(CASE WHEN o.shippedDate > o.requiredDate THEN 1 ELSE 0 END) / COUNT(*), 1) AS late_pct
FROM orders o
WHERE o.shippedDate IS NOT NULL
GROUP BY o.shipVia;
'''
run_query(q6)

**Q7 — Revenue by product category**

In [ ]:
q7 = '''
SELECT
    cat.categoryName,
    ROUND(SUM(od.unitPrice * od.quantity * (1 - od.discount)), 2) AS revenue
FROM order_details od
JOIN products p ON od.productID = p.productID
JOIN categories cat ON p.categoryID = cat.categoryID
GROUP BY cat.categoryName
ORDER BY revenue DESC;
'''
run_query(q7)

**Q8 — Month-over-month revenue growth % (window function: LAG)**

In [ ]:
q8 = '''
WITH monthly AS (
    SELECT
        strftime('%Y-%m', o.orderDate) AS month,
        SUM(od.unitPrice * od.quantity * (1 - od.discount)) AS revenue
    FROM orders o
    JOIN order_details od ON o.orderID = od.orderID
    GROUP BY month
)
SELECT
    month,
    ROUND(revenue, 2) AS revenue,
    ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY month)) / LAG(revenue) OVER (ORDER BY month), 1) AS mom_growth_pct
FROM monthly
ORDER BY month;
'''
run_query(q8)

**Q9 — Customers who ordered only once (churn risk / one-time buyers)**

In [ ]:
q9 = '''
SELECT
    c.companyName,
    c.country,
    COUNT(o.orderID) AS total_orders
FROM customers c
JOIN orders o ON c.customerID = o.customerID
GROUP BY c.customerID, c.companyName, c.country
HAVING COUNT(o.orderID) = 1
ORDER BY c.country;
'''
run_query(q9)

**Q10 — Top supplier by revenue contribution**

In [ ]:
q10 = '''
SELECT
    s.companyName AS supplier,
    ROUND(SUM(od.unitPrice * od.quantity * (1 - od.discount)), 2) AS revenue
FROM order_details od
JOIN products p ON od.productID = p.productID
JOIN suppliers s ON p.supplierID = s.supplierID
GROUP BY s.companyName
ORDER BY revenue DESC
LIMIT 10;
'''
run_query(q10)

## 6. Export Results for Excel & Power BI

Save key query outputs as CSVs so you can build pivot tables in Excel and connect Power BI directly to the same cleaned data / database file.

In [ ]:
queries = {
    'monthly_revenue': q1,
    'top_products': q2,
    'revenue_by_country': q3,
    'customer_rfm': q5,
}

EXCEL = os.path.join(BASE, 'excel')
for name, q in queries.items():
    df_result = run_query(q)
    out_path = os.path.join(EXCEL, f'{name}.csv')
    df_result.to_csv(out_path, index=False)
    print(f"Saved {name} -> {out_path}  ({len(df_result)} rows)")

conn.close()

## 7. Next Steps

1. Open `excel/monthly_revenue.csv`, `top_products.csv`, `revenue_by_country.csv`, `customer_rfm.csv` in Excel — build pivot tables, a pivot chart, and conditional formatting.
2. Open Power BI Desktop → Get Data → either the `northwind.db` SQLite file (needs an ODBC driver) or the cleaned CSVs in `data_clean/` and `excel/` → build KPI cards, trend line, country map, top products bar chart, and 1-2 DAX measures (YoY growth, running total).
3. Write your findings (see README template) and push everything to GitHub.